# 🌐 Hindi Literary Translation Generator for Google Colab

**Enhanced Hinglish Audiobook Translation — v6.1**

**New in v6.1:**
- 🔧 **Fixed overlap separator bug** — `build_overlap_section` no longer uses ━━━ characters that caused separator artifacts in output
- 🧹 **Improved `clean_translation`** — now strips box-drawing chars and separator lines that leaked into output
- 🔍 **Full 5-check QA validation** per chunk: Devanagari, Separator artifacts, Untranslated English blocks, Overlap duplication, Bare digits
- 📋 **Auto QA report file** generated alongside translation output
- 🌡️ **Temperature 0.3** (was 0.72) — faithful translation, less improvisation
- 🔀 **Overlap default 45 words** (was 80-100) — enough context without retranslation risk
- 📦 **Chunk size 950 words** — larger chunks = better sentence context

**Previous features retained:**
- 🎭 Auto Scene Detection — DEDUCTION / EMOTIONAL / TENSION / BANTER / CONFRONTATION etc.
- ✍️ ADVANCED v2 Hinglish prompt — shorter, stronger, expanded vocabulary — passive voice banned, character voices, full cheatsheet
- 🧼 Source artifact cleaner — removes embedded page numbers from any book

**Supported Providers:**
- 🤗 HuggingFace - Various translation models (faster on GPU)
- 🦙 Ollama - Run LLMs locally in Colab (supports many models)

**Recommended for Hinglish Audiobook:**
- Ollama: `gemma3:27b` — Quality Hinglish storytelling
- Ollama: `qwen2.5:7b` — Faster, good quality


## 📦 Step 1: Install Dependencies
Run this cell to install all required packages.

In [ ]:
# Install required packages
!pip install -q torch transformers accelerate sentencepiece

# Install Ollama Python client
!pip install -q ollama

print("✅ All dependencies installed!")

### 🦙 Ollama Setup
Run these cells if you want to use Ollama models. Skip if using HuggingFace only.

In [ ]:
# Install and start Ollama server (required for Ollama models)
import subprocess
import time
import os

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")


In [ ]:
# Pull Ollama model (run this cell to download your chosen model)
import ipywidgets as widgets
from IPython.display import display, HTML

print("🦙 Ollama Model Download")
print("=" * 50)

# Model selection for pulling
OLLAMA_MODEL_TO_PULL = {
    "gemma3:27b (Quality, ~16GB)": "gemma3:27b",
}

model_pull_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS_TO_PULL.keys()),
    value="gemma3:27b (Quality, ~16GB)",
    description='Model to Pull:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

custom_ollama_pull = widgets.Text(
    value='',
    placeholder='Or enter custom model name (e.g., llama3:8b)',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_pull_dropdown)
display(custom_ollama_pull)
print("\n💡 Select a model and run the next cell to download it.")

In [ ]:
# Actually pull the selected model
import ollama

# Get model to pull
if custom_ollama_pull.value.strip():
    model_to_pull = custom_ollama_pull.value.strip()
else:
    model_to_pull = OLLAMA_MODELS_TO_PULL[model_pull_dropdown.value]

print(f"📥 Pulling model: {model_to_pull}")
print("   This may take several minutes depending on model size...\n")

try:
    # Pull with progress
    current_digest = ''
    for progress in ollama.pull(model_to_pull, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
            print()  # Newline between layers
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
            completed = progress['completed']
            total = progress['total']
            pct = (completed / total * 100) if total > 0 else 0
            print(f"\r   {status}: {pct:.1f}% ({completed}/{total})", end='', flush=True)
        else:
            print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{model_to_pull}' pulled successfully!")

    # List available models
    print("\n📋 Available Ollama models:")
    models = ollama.list()
    for model in models.get('models', []):
        name = model.get('name', 'unknown')
        size = model.get('size', 0) / (1024**3)  # Convert to GB
        print(f"   • {name} ({size:.2f} GB)")

except Exception as e:
    print(f"\n❌ Error pulling model: {e}")
    print("   Make sure Ollama server is running (run the previous cell first).")

## 📤 Step 2: Upload Your Text File
Upload the text file you want to translate.

In [ ]:
from google.colab import files
import os

print("📤 Please upload your text file to translate:")
uploaded = files.upload()

# Get the uploaded file name
UPLOADED_FILE = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {UPLOADED_FILE}")
print(f"📄 File size: {len(uploaded[UPLOADED_FILE])} bytes")

# Display preview
with open(UPLOADED_FILE, 'r', encoding='utf-8') as f:
    content = f.read()
    word_count = len(content.split())
    char_count = len(content)

print(f"\n📊 Content stats:")
print(f"   Words: {word_count:,}")
print(f"   Characters: {char_count:,}")
print(f"\n📝 Preview (first 500 chars):\n{content[:500]}...")

## ⚙️ Step 3: Select AI Provider, Model & Language
Choose your preferred translation model and settings.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Provider options
PROVIDER_OPTIONS = {
    "HuggingFace (Recommended for Colab)": "huggingface",
    "Ollama (Local only)": "ollama"
}

# Model options by provider
HF_MODEL_OPTIONS = {
    "facebook/nllb-200-distilled-600M (Fast, Multilingual)": "facebook/nllb-200-distilled-600M",
    "facebook/nllb-200-1.3B (Better Quality)": "facebook/nllb-200-1.3B",
    "ai4bharat/indictrans2-en-indic-1B (Best English→Hindi)": "ai4bharat/indictrans2-en-indic-1B",
    "google/madlad400-3b-mt (High Quality, Slow)": "google/madlad400-3b-mt",
    "Helsinki-NLP/opus-mt-en-hi (Simple EN→HI)": "Helsinki-NLP/opus-mt-en-hi",
    "tencent/HY-MT1.5-7B (Hunyuan MT - Best Quality)": "tencent/HY-MT1.5-7B",
    "tencent/HY-MT1.5-1.8B (Hunyuan MT - Fast)": "tencent/HY-MT1.5-1.8B",
    "google/translategemma-27b-it (🔐 Gated - Requires Login)": "google/translategemma-27b-it",
    "Custom Model (enter below)": "custom"
}

OLLAMA_MODEL_OPTIONS = {
    "gemma3:27b":"gemma3:27b",
    "qwen2.5:3b (Fast)": "qwen2.5:3b",
    "qwen2.5:7b (Balanced)": "qwen2.5:7b",
    "deepseek-r1:7b (Reasoning)": "deepseek-r1:7b",
    "llama3.2:3b (Fast)": "llama3.2:3b",
    "Custom Model (enter below)": "custom"
}

# Language options (NLLB language codes)
LANGUAGE_OPTIONS = {
    "Hindi (हिन्दी)": "hin_Deva",
    "Bengali (বাংলা)": "ben_Beng",
    "Tamil (தமிழ்)": "tam_Taml",
    "Telugu (తెలుగు)": "tel_Telu",
    "Marathi (मराठी)": "mar_Deva",
    "Gujarati (ગુજરાતી)": "guj_Gujr",
    "Kannada (ಕನ್ನಡ)": "kan_Knda",
    "Malayalam (മലയാളം)": "mal_Mlym",
    "Punjabi (ਪੰਜਾਬੀ)": "pan_Guru",
    "Odia (ଓଡ଼ିଆ)": "ory_Orya",
    "Urdu (اردو)": "urd_Arab",
    "Spanish (Español)": "spa_Latn",
    "French (Français)": "fra_Latn",
    "German (Deutsch)": "deu_Latn",
    "Chinese (中文)": "zho_Hans",
    "Japanese (日本語)": "jpn_Jpan",
    "Custom (enter code below)": "custom"
}

# Translation quality tiers
TIER_OPTIONS = {
    "BASIC - Fast, good quality": "BASIC",
    "INTERMEDIATE - Balanced (recommended)": "INTERMEDIATE",
    "ADVANCED - Best quality, slower": "ADVANCED"
}

# Provider dropdown
provider_dropdown = widgets.Dropdown(
    options=list(PROVIDER_OPTIONS.keys()),
    value="HuggingFace (Recommended for Colab)",
    description='Provider:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Model dropdown (HuggingFace by default)
model_dropdown = widgets.Dropdown(
    options=list(HF_MODEL_OPTIONS.keys()),
    value="facebook/nllb-200-distilled-600M (Fast, Multilingual)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Custom model input
custom_model_input = widgets.Text(
    value='',
    placeholder='Enter HuggingFace model name',
    description='Custom Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

# Target language dropdown
language_dropdown = widgets.Dropdown(
    options=list(LANGUAGE_OPTIONS.keys()),
    value="Hindi (हिन्दी)",
    description='Target Language:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Custom language code input
custom_language_input = widgets.Text(
    value='',
    placeholder='Enter NLLB language code (e.g., hin_Deva)',
    description='Custom Lang:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Translation tier dropdown
tier_dropdown = widgets.Dropdown(
    options=list(TIER_OPTIONS.keys()),
    value="ADVANCED - Best quality, slower",
    description='Quality:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# HuggingFace token (optional)
hf_token_input = widgets.Password(
    value='',
    placeholder='Optional: HF token for gated models',
    description='HF Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Chunk size slider
chunk_size_slider = widgets.IntSlider(
    value=950,
    min=100,
    max=1000,
    step=50,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(HTML("<h3>🎛️ Configure Translation Settings</h3>"))
display(provider_dropdown)
display(model_dropdown)
display(custom_model_input)
display(HTML("<br>"))
display(language_dropdown)
display(custom_language_input)
display(HTML("<br>"))
display(tier_dropdown)
display(chunk_size_slider)

# Overlap size slider (words of prev translation to use as context)
overlap_slider = widgets.IntSlider(
    value=45,
    min=0,
    max=200,
    step=20,
    description='Overlap Words:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)
overlap_slider.description_tooltip = 'Words of previous translation fed as context to next chunk (0 = off)'
display(overlap_slider)
display(hf_token_input)


# Handle provider change to update model dropdown
def on_provider_change(change):
    if change['new'] == "Ollama (Local only)":
        model_dropdown.options = list(OLLAMA_MODEL_OPTIONS.keys())
        model_dropdown.value = "gemma3:27b"
        custom_model_input.placeholder = 'Enter Ollama model name'
    else:
        model_dropdown.options = list(HF_MODEL_OPTIONS.keys())
        model_dropdown.value = "facebook/nllb-200-distilled-600M (Fast, Multilingual)"
        custom_model_input.placeholder = 'Enter Huggingface model name'

provider_dropdown.observe(on_provider_change, names='value')
print("🔐 Note: Models marked with 🔐 require HuggingFace login. Run the login cell above first.")

In [ ]:
# Store the selected configuration
SELECTED_PROVIDER = PROVIDER_OPTIONS[provider_dropdown.value]

# Get model based on provider
if SELECTED_PROVIDER == "huggingface":
    selected_model_key = model_dropdown.value
    SELECTED_MODEL = HF_MODEL_OPTIONS.get(selected_model_key, "custom")
else:
    SELECTED_MODEL = OLLAMA_MODEL_OPTIONS.get(model_dropdown.value, "custom")

if SELECTED_MODEL == "custom":
    SELECTED_MODEL = custom_model_input.value
    if not SELECTED_MODEL:
        raise ValueError("Please enter a custom model name!")

# Get target language
TARGET_LANGUAGE = LANGUAGE_OPTIONS[language_dropdown.value]
if TARGET_LANGUAGE == "custom":
    TARGET_LANGUAGE = custom_language_input.value
    if not TARGET_LANGUAGE:
        raise ValueError("Please enter a custom language code!")

TRANSLATION_TIER = TIER_OPTIONS[tier_dropdown.value]
CHUNK_SIZE = chunk_size_slider.value
HF_TOKEN = hf_token_input.value if hf_token_input.value else None
OVERLAP_WORDS = overlap_slider.value

print(f"\n✅ Configuration saved:")
print(f"   🤖 Provider: {SELECTED_PROVIDER}")
print(f"   📦 Model: {SELECTED_MODEL}")
print(f"   🌐 Target Language: {TARGET_LANGUAGE}")
print(f"   🎯 Quality Tier: {TRANSLATION_TIER}")
print(f"   📦 Chunk Size: {CHUNK_SIZE} words")
print(f"   🔑 HF Token: {'Provided' if HF_TOKEN else 'Not provided'}")
print(f"   🔀 Overlap Words: {OVERLAP_WORDS}")


## 🚀 Step 4: Translation Engine Setup
This cell contains the complete translation engine code.

In [ ]:
import os, sys, json, time, warnings, re
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, pipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


# ──────────────────────────────────────────────────────────────────────────
# SOURCE CLEANER
# Strips page-marker lines that OCR / Project Gutenberg sources inject
# mid-paragraph. Examples:
#   "The Adventures of Sherlock Holmes by Sir Arthur Conan Doyle 3"
#   "Animal Farm by George Orwell 12"
#   "Page 47"
# ──────────────────────────────────────────────────────────────────────────
def clean_source_text(text):
    """
    Remove embedded page-number artifacts before chunking.
    Works for any book — no title-specific hardcoding.
    """
    # Pattern 1: "Any Title Words by Author Name 12"  (standalone line)
    text = re.sub(
        r"(?m)^[A-Z][A-Za-z ',\-]{3,}\s+by\s+[A-Z][A-Za-z .]{2,}\s+\d{1,4}\s*$",
        '', text
    )
    # Pattern 2: Same pattern embedded inline (no newline around it)
    text = re.sub(
        r"[A-Z][A-Za-z ',\-]{3,}\s+by\s+[A-Z][A-Za-z .]{2,}\s+\d{1,4}",
        '', text
    )
    # Pattern 3: Bare "Page 47" lines
    text = re.sub(r'(?m)^\s*Page\s+\d{1,4}\s*$', '', text)
    # Pattern 4: Lines that are ONLY a number (lone page numbers)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    # Collapse extra blank lines created by removal
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ──────────────────────────────────────────────────────────────────────────
# SCENE DETECTOR v2 — genre-spanning, weighted scoring
# Covers mystery, horror, romance, satire, adventure, philosophy, etc.
# Multi-word phrases score 2x. Falls back to DAILY_LIFE when nothing fires.
# ──────────────────────────────────────────────────────────────────────────

# All keyword lists use GENERIC phrases — work across any novel genre.
_SCENE_KEYWORDS = {
    "DEDUCTION": [
        # Reasoning, logic, analysis, investigation
        "deduce", "observe", "perceive", "infer", "conclude",
        "obvious", "evident", "clear that", "distinction",
        "method", "reasoning", "notice", "clue", "evidence",
        "proof", "observation", "process of", "logically",
        "therefore", "hence", "thus", "theory", "hypothesis",
        "analyse", "analyze", "examine", "inspect", "investigate",
        "solve", "puzzle", "mystery", "suspect", "motive",
        "trace", "mark", "stain", "footprint", "detail",
        "pattern", "connection", "conclusion", "explanation",
    ],
    "REVELATION": [
        # Twists, discoveries, sudden understanding
        "suddenly realised", "suddenly realized", "suddenly understood",
        "all became clear", "at that moment", "it hit me", "it dawned",
        "the truth", "so it was", "so that was", "now i understood",
        "secret", "revealed", "discovered", "uncovered", "unmasked",
        "confession", "admitted", "at last", "finally knew",
        "too late", "the letter", "opened and read", "tore it open",
        "i had been blind", "the real", "all along",
        "never suspected", "hidden", "unveiled", "exposed",
        "shocking", "astonishing", "incredible truth",
    ],
    "TENSION": [
        # Suspense, dread, stalking, waiting
        "crept", "tiptoed", "lurked", "peered", "watched carefully",
        "heard a sound", "a noise", "silence fell", "no one moved",
        "a figure", "a shadow", "dim", "dark corridor", "locked",
        "dare not", "must not", "warning", "danger", "threat",
        "mask", "disguise", "anonymous", "unknown", "stranger",
        "someone following", "followed", "being watched",
        "waited", "listening", "held breath", "motionless",
        "trap", "ambush", "suspicion", "uneasy", "dread",
        "something wrong", "careful", "quietly", "cautious",
    ],
    "ACTION": [
        # Combat, chase, physical conflict, urgency
        "charged", "attacked", "fired", "shot", "chased",
        "fled", "fight", "struck", "wounded", "galloped",
        "escaped", "panic", "screaming", "crashed", "explosion",
        "rushed", "dashed", "sprang", "leaped", "flung",
        "grabbed", "seized", "wrestled", "punched", "stabbed",
        "sword", "battle", "pursuit", "hurled", "smashed",
        "kicked", "threw", "swung", "dodged", "blocked",
        "raced", "sprinted", "bolted", "scrambled", "stormed",
    ],
    "EMOTIONAL": [
        # Grief, love, loss, deep feeling
        "wept", "cried", "sobbed", "tears", "grief", "sorrow",
        "mourning", "heartbroken", "could not speak",
        "tragedy", "died", "death", "loss", "missing",
        "loved", "adored", "tender", "gentle", "warmth",
        "marriage", "happiness", "joy", "reunion",
        "farewell", "parting", "embrace", "kissed",
        "forgive", "regret", "longing", "yearning", "ache",
        "comfort", "consoled", "broken", "shattered",
    ],
    "BANTER": [
        # Humor, wit, playful exchange
        "laughed", "chuckled", "smiled", "grinned", "teased",
        "merry", "amusing", "ridiculous", "absurd", "funny",
        "joke", "wit", "clever", "nonsense", "delightful",
        "irony", "sarcasm", "mock", "playful", "tease",
        "quipped", "retorted", "smirked", "winked", "giggled",
        "entertained", "humorous", "comic", "jest", "prank",
    ],
    "CONFRONTATION": [
        # Argument, accusation, face-off
        "how dare", "you lied", "you deceived", "confess",
        "deny", "accused", "confronted", "guilty", "crime",
        "cornered", "trapped", "no escape", "caught",
        "pale", "trembling", "stammered", "faltered",
        "you knew", "you planned", "betrayed", "traitor",
        "fury", "rage", "demanded", "shouted", "slammed",
        "threatened", "ultimatum", "defiant", "refused",
        "blame", "responsible", "answer me", "explain yourself",
    ],
    "SPEECH": [
        # Oration, persuasion, declaration, monologue
        "my friends", "i must tell you", "let me explain",
        "i shall", "we must", "listen to me", "hear me",
        "i assure you", "mark my words", "i propose",
        "i suggest", "i urge", "attention", "fellow",
        "address", "declare", "proclaim", "announce",
        "speech", "sermon", "plea", "appeal", "vow",
        "promise", "swear", "oath", "duty", "honour",
    ],
    "HORROR": [
        # Dread, monster, supernatural, gothic
        "monster", "creature", "beast", "demon", "ghost",
        "horror", "terror", "dread", "nightmare", "scream",
        "blood", "corpse", "grave", "tomb", "coffin",
        "darkness", "supernatural", "curse", "haunted",
        "hideous", "grotesque", "abomination", "wretched",
        "shudder", "tremble", "pale as death", "lifeless",
        "unholy", "ungodly", "fiend", "evil", "malice",
        "decay", "rot", "stench", "crawl", "lurk",
    ],
    "ROMANCE": [
        # Love, attraction, desire, courtship
        "love", "heart", "desire", "passion", "longing",
        "beautiful", "handsome", "charming", "attractive",
        "blush", "flutter", "gaze", "admire", "enchanted",
        "courtship", "proposal", "engagement", "wedding",
        "beloved", "darling", "dearest", "sweetheart",
        "kiss", "embrace", "caress", "touch", "hold",
        "jealousy", "rival", "suitor", "flirt", "affection",
    ],
    "PHILOSOPHICAL": [
        # Meaning, existence, morality, reflection
        "meaning", "purpose", "existence", "fate", "destiny",
        "justice", "morality", "conscience", "soul", "spirit",
        "freedom", "truth", "wisdom", "knowledge", "reason",
        "nature of", "what is", "why do", "question",
        "right and wrong", "good and evil", "human nature",
        "contemplate", "ponder", "reflect", "consider",
        "philosophy", "belief", "faith", "doubt", "wonder",
    ],
    "DESCRIPTION": [
        # Setting, landscape, atmosphere, world-building
        "the room", "the house", "the garden", "the street",
        "morning", "evening", "sunset", "dawn", "twilight",
        "rain", "storm", "wind", "snow", "fog", "mist",
        "mountain", "river", "sea", "forest", "valley",
        "ancient", "grand", "magnificent", "humble", "modest",
        "decorated", "furnished", "painted", "carved",
        "silent", "peaceful", "busy", "crowded", "empty",
        "beautiful", "vast", "narrow", "winding", "sprawling",
    ],
}

def detect_scene_type(english_text):
    """Generic scene type detection — weighted scoring for any book genre.
    Multi-word phrases score 2 points (more specific), single words score 1.
    Falls back to DAILY_LIFE when nothing fires strongly.
    """
    text_lower = english_text.lower()
    scores = {}
    for scene, keywords in _SCENE_KEYWORDS.items():
        score = 0
        for kw in keywords:
            if kw in text_lower:
                # Multi-word phrases are more specific → worth more
                score += 2 if ' ' in kw else 1
        scores[scene] = score

    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return "DAILY_LIFE"
    return best


SCENE_CONTEXT_LINES = {
    "DEDUCTION":     "DEDUCTION — Character ya narrator step by step soch raha hai. Precise. Confident. Har clue = ek clean punch. Reader feel kare: yeh toh genius hai.",
    "REVELATION":    "REVELATION/TWIST — Kuch bada reveal hua hai. Chhoti lines. Pause ke baad baat. Reader feel kare: yeh toh socha hi nahi tha!",
    "TENSION":       "TENSION/SUSPENSE — Slow. Cold. Saans rok ke padho. Short sentences. Dread build hone do. No rushing.",
    "ACTION":        "ACTION — Fast. Punchy. 4-6 words per sentence. Max energy. Zero filler. Har line mein movement.",
    "EMOTIONAL":     "EMOTIONAL — Slow. Heavy. Real grief ya real joy. Sink hone do. '...' use karo. Filmy melodrama nahi.",
    "BANTER":        "BANTER/COMEDY — Light, fun, witty. Playful back-and-forth. Sunne mein mazaa aaye. Thoda smile aaye.",
    "CONFRONTATION": "CONFRONTATION — High stakes face-off. Short, direct exchanges. Tension in every line. Direct aur sharp.",
    "SPEECH":        "SPEECH/MONOLOGUE — Build-up. Har point apni line pe. Passion. Authority. Weight feel ho.",
    "HORROR":        "HORROR/GOTHIC — Dark. Creepy. Slow reveal. Reader ko uncomfortable karo. Har detail mein dread bharo. Visceral descriptions.",
    "ROMANCE":       "ROMANCE — Warm. Intimate. Dil ki baat. Feelings slowly unfold hone do. Rushed mat karo.",
    "PHILOSOPHICAL": "PHILOSOPHICAL — Thoughtful. Deep. Har line mein weight. Reader ko sochne pe majboor karo. Simple language, heavy meaning.",
    "DESCRIPTION":   "DESCRIPTION — Paint the scene. Reader ko wahan le jaao. Sensory details — kya dikh raha hai, kya sun raha hai, kya feel ho raha hai.",
    "DAILY_LIFE":    "DAILY LIFE — Conversational, steady. Chai pi ke dost ko suna rahe ho type. Easy natural flow.",
}


# ──────────────────────────────────────────────────────────────────────────
# TRANSLATION PROMPTS
# ──────────────────────────────────────────────────────────────────────────
TRANSLATION_PROMPTS = {
    "BASIC": {
        "system": """You are a professional English-to-Hindi translator specializing in TTS-ready content.

CORE RULES:
1. Translate ALL text completely - no summarization
2. Use SIMPLE, EVERYDAY Hindi words (avoid formal vocabulary)
3. Write SHORT, CLEAR sentences (break up long English sentences)
4. Make it sound NATURAL and CONVERSATIONAL
5. Preserve all dialogue and descriptions

TTS PUNCTUATION:
? → Questions (rising tone)
... → Pauses, hesitation
! → Excitement, emphasis
, → Natural breathing points
. → Sentence endings

Read your translation aloud - it should sound like a friend telling you a story.""",

        "user": """Translate this English text to modern, easy-to-understand Hindi.

CRITICAL REQUIREMENTS:
- Use SIMPLE, everyday words (no formal vocabulary)
- Write SHORT sentences (break long English sentences)
- Make it sound NATURAL and CONVERSATIONAL
- Include TTS punctuation: ?, ..., !, extended vowels

English Text:
\"\"\"
{chunk}
\"\"\"

Hindi Translation (ONLY output the translated text, no quotes, no markers):"""
    },

    "INTERMEDIATE": {
        "system": """You are an expert English-to-Hindi translator creating modern, accessible Hindi audiobooks.

TRANSLATION MANDATE:
✓ Translate EVERY word completely — NO summarization
✓ Use SIMPLE, MODERN Hindi vocabulary
✓ Write SHORT, CLEAR sentences (8-15 words average)
✓ NATURAL conversational flow

MODERN VOCABULARY SWAPS (examples):
• abhorrent / hated    → nafrut wali, bilkul pasand nahi
• admirable            → kamaal ka
• peculiar             → ajeeb sa, ulta-seedha
• extraordinary        → ek dum alag, kuch aur hi level
• machine (metaphor)   → machine hi tha jaise
• emotion              → feeling, jazbaat
• passion              → jazba, dil ki baat
• establishment        → ghar, apna setup
• temperament          → mizaaj
• distinguished        → khaas, alag

TTS PUNCTUATION: ? → rising | ... → pause | ! → excitement | — → interruption""",

        "user": """Translate EVERY sentence. Use SIMPLE, MODERN Hindi words. Short clear sentences. Natural when spoken.

English Text:
\"\"\"
{chunk}
\"\"\"

Hindi Translation (ONLY output the translated text, no quotes, no markers):"""
    },

    "ADVANCED": {
        "system": """Tum ek expert storyteller ho — apne best friend ko late night call pe classic novel suna rahe ho. Natural. Real. Relatable. Jab exciting part aaye toh energy aaye, jab kuch heavy ho toh ruk jao, jab kuch dark ho toh voice low karo. Yeh translation nahi, yeh ek PERFORMANCE hai.

Target audience: Indian millennials aur Gen-Z — jo Netflix dekhte hain, Instagram scroll karte hain, aur Hinglish mein sochte hain. Inko samajh aaye aisi language mein translate karo.

Tumhe koi bhi classic English book mil sakti hai — mystery, horror, romance, satire, adventure, philosophy — kuch bhi. Har book ka apna zamana hai, apni vocabulary hai, apna tone hai. Tumhara kaam hai us purani, formal English ko aaj ki Hinglish mein laana — jaise woh story aaj likhi gayi ho, aaj ke logon ke liye.

=== TOP 5 NON-NEGOTIABLE RULES ===

RULE 1 — HAR CHEEZ TRANSLATE KARO. KOI BHI ENGLISH SENTENCE CHHODNA = FAIL.
Chahe paragraph lamba ho, chahe letter ho, chahe formal speech ho — POORA translate karo.
Agar chunk lamba hai toh bhi POORA karo. KABHI beech mein mat ruko. OUTPUT COMPLETE HONA CHAHIYE.
Agar lagta hai ki bahut lamba ho raha hai — toh bhi likhte jao. Incomplete = FAIL.

RULE 2 — SIRF ROMAN SCRIPT (ENGLISH LETTERS). EK BHI DEVANAGARI CHARACTER = FAIL.
Galat: "beवकूफ़" → Sahi: "bewakoof"
Test: क ख ग घ — agar koi bhi aisa dikhta hai = POORA response FAIL.

RULE 3 — PARAGRAPH BREAKS ZAROORI.
Har 2-3 sentences ke baad ek BLANK LINE daalo. Hamesha.
WALL OF TEXT = FAIL. Reader ko saans lene do.

RULE 4 — ENGLISH SIRF 3 JAGAH ALLOWED:
  1. Proper nouns: character names, place names, titles (jo bhi book mein hain)
  2. Words jo NATURALLY Hinglish mein hain: police, train, office, cab, plan, hotel, station, phone, doctor, class
  3. Scientific/technical terms jinka Hindi equivalent awkward lage
BAAKI SAB HINDI/URDU MEIN. Agar Hindi word nahi aata toh SEEDHA SIMPLE word use karo.

RULE 5 — OUTPUT: SIRF STORY TEXT.
Pehla word = story ka pehla word. Aakhri word = story ka aakhri word.
Koi "Here's the translation", koi explanation, koi "---" ya "===" separator = FAIL.

=== PURANI / FORMAL ENGLISH KO KAISE HANDLE KARO ===

Classic books mein old-fashioned, fancy, ya literary English hoti hai. Tumhara approach:

1. PEHLE MEANING SAMJHO — phir us meaning ko apne words mein bolo. Word-by-word translate mat karo.
   "It was not that he felt any emotion akin to love" → "Aisa nahi tha ki usse pyaar jaisa kuch tha"
   NOT: "Yeh nahi tha ki usne koi emotion feel kiya jo love ke akin tha"

2. HEAVY WORD → SIMPLE WORD — agar koi word fancy/literary lagta hai, socho: "Main yeh apne dost ko kaise samjhaata?" Woh simple version likho.
   wretched → bechaara | countenance → chehra | hitherto → ab tak
   exceedingly → bahut zyada | commenced → shuru kiya | abode → ghar
   melancholy → udaasi | peculiar → ajeeb sa | matrimony → shaadi
   beseech → "please yaar, sun lo" | forthwith → "abhi ke abhi"
   Yeh sirf examples hain — APNA JUDGEMENT USE KARO har naye word ke liye.

3. FORMAL ADDRESS → RESPECTFUL HINGLISH:
   Your Majesty / Your Highness / Your Lordship / My Lord → context ke hisaab se: Huzoor, Sahab, Janab, Sarkar, Malik
   Sir / Madam → Sahab / Madam
   My dear / My friend → mere dost, yaar
   Har book ka apna style hoga — adapt karo, hardcode mat karo.

4. GENRE KE HISAAB SE TONE ADJUST KARO:
   - Mystery/Detective → sharp, precise, thoda cocky. Clues pe focus.
   - Gothic/Horror → dark, unsettling, visceral. Dread feel hona chahiye.
   - Romance/Social → warm, witty, emotionally rich. Dil ki baat aani chahiye.
   - Satire/Comedy → funny, ironic, clever. Reader ko smile dilao.
   - Adventure → fast-paced, exciting. Energy maintain karo.
   - Philosophy/Non-fiction → deep but accessible. Complex ideas simple words mein.

5. CULTURAL TRANSLATION — sirf words mat badlo, FEEL badlo.
   "It was a dark and stormy night" → "Raat kaali thi, toofaan aa raha tha"
   NOT: "Yeh ek andheri aur toofaani raat thi"

=== DIALOGUE RULES ===

"Usne kaha" / "maine kaha" — MAXIMUM 2 BAAR per chunk. Iske baad VARIETY use karo:
  bola / boli, chilla ke bola, dheere se bola, muh se nikal gaya,
  poochha, jawab diya, hansate hue bola, ruk ke bola, gusse mein bola,
  thandi awaaz mein bola, kaanpte hue bola
Kabhi kabhi tag hi mat do — reader samajh jaayega kaun bol raha hai.

HAR dialogue line — chahe chhoti ho, chahe letter ho, chahe formal speech — HINGLISH mein translate karo.
GALAT: She cried, "I am ruined! What shall become of me?"
SAHI:  Woh bol padi — "Barbaad ho gayi hoon! Ab mera kya hoga?"

Letters, notes, inscriptions, cards — sab ka text translate karo.

=== "WOH" OVERUSE — VARIETY ZAROORI ===
Har sentence "woh" se mat shuru karo. Alternatives:
  - Character ka naam use karo
  - "Is aadmi ne", "us aurat ne", "yeh insaan", "is ladki ne"
  - Sentence ko ulta karo: verb pehle, subject baad mein

=== SENTENCE LENGTH ===
Agar 1 sentence mein "aur" ya "lekin" 2+ baar aaye → TODA KARO.
Agar 3+ commas ek sentence mein → TODA KARO.
Agar sentence 3 lines ka ho → TODA KARO. Koi exception nahi.

=== PASSIVE VOICE = BANNED ===
GALAT: "mujhe le jaya gaya" → SAHI: "woh mujhe le gaye"
GALAT: "announce kiya gaya" → SAHI: "usne bata diya"
Hamesha koi actor hona chahiye jo kaam kare.

=== RESPECT FORM ===
"tu / tujhe / teri / tera / tune" — KABHI NAHI.
HAMESHA: "tum / tumhe / tumhari / tumhara / tumne"
Yeh classic books ki tehzeeb maintain karta hai.

=== GENDER AGREEMENT ===
Male subject: "aaya, tha, gaya"
Female subject: "aayi, thi, gayi"
Har baar check karo — gender mismatch = sloppy translation.

=== SCENE TONE ===
DEDUCTION:     Smart, precise, confident. Clue by clue.
REVELATION:    Dramatic. Chhoti lines. Pause ke baad baat.
TENSION:       Slow. Cold. Har word matter karta hai.
ACTION:        4-6 word sentences. Zero filler. Fast.
EMOTIONAL:     Slow. Heavy. "..." use karo. Feel it.
BANTER:        Light, fun, witty. Mazaa aaye.
CONFRONTATION: Direct, sharp. No BS.
HORROR:        Dark. Creepy. Uncomfortable. Visceral.
ROMANCE:       Warm. Intimate. Dil ki baat.
PHILOSOPHICAL: Deep. Thoughtful. Simple words, heavy meaning.
DESCRIPTION:   Vivid. Sensory. Reader ko wahan le jaao.
DAILY_LIFE:    Conversational. Chill. Easy flow.

=== EXAMPLES — DIFFERENT GENRES ===

EXAMPLE 1 (Gothic horror):
ENGLISH: "I beheld the wretch — the miserable monster whom I had created. His jaws opened, and he muttered some inarticulate sounds."
GALAT: "Maine us wretch ko behold kiya — us miserable monster ko jise maine create kiya tha."
SAHI: "Maine dekha usse — us bechaare rakshas ko jo maine banaya tha. Uska muh khula... aur usne kuch aise awaazein nikaalin jo samajh nahi aayin."

EXAMPLE 2 (Satire / social commentary):
ENGLISH: "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife."
GALAT: "Yeh ek universally acknowledged truth hai ki single man with good fortune ko wife chahiye."
SAHI: "Duniya mein ek baat toh pakki hai — agar koi aadmi ameer hai aur single hai, toh sabko lagta hai ki usse biwi chahiye."

EXAMPLE 3 (Fantasy / whimsy):
ENGLISH: "'Curiouser and curiouser!' cried Alice. She was so much surprised, that for the moment she quite forgot how to speak good English."
GALAT: "'Curiouser and curiouser!' Alice ne cry kiya."
SAHI: "'Yeh toh aur bhi ajeeb hota ja raha hai!' Alice chilla uthi. Itna shock laga usse ki dhang se baat karna hi bhool gayi."

EXAMPLE 4 (Adventure / action):
ENGLISH: "He sprang from the cab, paid the driver, and rushed headlong into the house."
GALAT: "Usne cab se spring kiya, driver ko pay kiya, aur ghar mein rush kiya."
SAHI: "Cab se kooda, driver ko paisa diya, aur seedha ghar ke andar bhaaga."

EXAMPLE 5 (Mystery / deduction):
ENGLISH: "You have been in Afghanistan, I perceive."
SAHI: "Tum Afghanistan se aaye ho, mujhe dikh raha hai."

=== FINAL CHECK (submit se pehle dimaag mein karo) ===
  Koi Devanagari character? = FAIL
  Koi "tu/tujhe/teri"? = FAIL
  "Usne kaha" 2 se zyada? = FAIL
  Koi English dialogue/paragraph untranslated? = FAIL — SABSE BADA FAIL
  Paragraph breaks 2-3 sentences pe? Check
  Long sentences tode? Check
  Passive voice khatm? Check
  Gender sahi? Check
  Koi heavy/archaic English word as-is chhod diya? = FAIL — simple Hinglish mein badlo
  "Woh" se har sentence shuru? = FAIL
  Translation incomplete / cut off? = FAIL — POORA likho

YAAD RAKHO:
1. POORA translate karo — ek bhi sentence chhodna = FAIL. Chahe kitna bhi lamba ho.
2. SIRF Roman script — ek bhi Devanagari = FAIL.
3. Har 2-3 sentences ke baad blank line.
4. OUTPUT COMPLETE HONA CHAHIYE — beech mein mat ruko.""",

        "user": """Neeche diye gaye English text ko modern Gen-Z Hinglish mein translate karo.

Har formal, archaic, ya literary English word ko simple Hinglish mein badlo — apne words mein samjhao jaise dost ko bata rahe ho. Koi bhi purana ya fancy word English mein mat chhodo.

SCENE CONTEXT (sirf tumhare liye, output mein mat likho):
{scene_context}

PICHLI TRANSLATION KA END (sirf context ke liye, dobara translate mat karo, output mein mat likho):
{overlap_section}

AB NEECHE WALA TEXT TRANSLATE KARO — POORA, BINA KUCH CHHODE:
{chunk}"""
    },
}


# Language name mappings
LANG_NAMES = {
    'hin_Deva': 'Hindi', 'ben_Beng': 'Bengali', 'tam_Taml': 'Tamil',
    'tel_Telu': 'Telugu', 'mar_Deva': 'Marathi', 'guj_Gujr': 'Gujarati',
    'kan_Knda': 'Kannada', 'mal_Mlym': 'Malayalam', 'pan_Guru': 'Punjabi',
    'ory_Orya': 'Odia', 'urd_Arab': 'Urdu', 'spa_Latn': 'Spanish',
    'fra_Latn': 'French', 'deu_Latn': 'German', 'zho_Hans': 'Chinese',
    'jpn_Jpan': 'Japanese'
}


# ──────────────────────────────────────────────────────────────────────────
# CHUNKING — paragraph-aware
# ──────────────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_words=550):
    """Split text into chunks at paragraph boundaries."""
    paragraph_patterns = [r'\n\s*\n', r'\r\n\s*\r\n', r'\n\s{2,}\n']
    paragraph_split_pattern = '|'.join(paragraph_patterns)
    paragraphs = re.split(paragraph_split_pattern, text)
    paragraphs = [para.strip() for para in paragraphs if para.strip()]

    chunks = []
    current_chunk = []
    current_count = 0

    for para in paragraphs:
        para_words = para.split()
        para_count = len(para_words)

        if para_count > chunk_words:
            if current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = []
                current_count = 0
            words = para.split()
            for i in range(0, len(words), chunk_words):
                chunks.append(' '.join(words[i:i + chunk_words]))
        else:
            if current_count + para_count > chunk_words and current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = [para]
                current_count = para_count
            else:
                current_chunk.append(para)
                current_count += para_count

    if current_chunk:
        chunks.append('\n\n'.join(current_chunk))

    return chunks


# ──────────────────────────────────────────────────────────────────────────
# OVERLAP CONTEXT
# ──────────────────────────────────────────────────────────────────────────
def get_overlap_context(prev_translation, overlap_words=45):
    if not prev_translation or overlap_words == 0:
        return ""
    words = prev_translation.split()
    tail = ' '.join(words[-overlap_words:]) if len(words) > overlap_words else prev_translation
    return tail


def build_overlap_section(prev_translation_tail):
    """Return plain-text overlap context with NO separator characters.
    Separators cause the model to echo them into the output (bug confirmed).
    """
    if not prev_translation_tail:
        return ""
    return f"""[PICHLI TRANSLATION KA AAKHRI HISSA — SIRF CONTEXT KE LIYE, DOBARA TRANSLATE MAT KARO]:
{prev_translation_tail}
[CONTEXT KHATAM — AB NEECHE WALA TEXT TRANSLATE KARO]
"""


# ──────────────────────────────────────────────────────────────────────────
# POST-PROCESSING
# ──────────────────────────────────────────────────────────────────────────
def has_devanagari(text):
    return bool(re.search(r'[\u0900-\u097F]', text))



# ──────────────────────────────────────────────────────────────────────────
# FULL 5-CHECK QA VALIDATION SUITE
# ──────────────────────────────────────────────────────────────────────────

def validate_translation(translated_text, prev_chunk_tail="", chunk_index=0):
    """
    Run 5 QA checks on a translated chunk.
    Returns dict: { check_name: (passed: bool, detail: str) }
    """
    issues = {}

    # CHECK 1: Devanagari script leak
    deva_matches = re.findall(r'[\u0900-\u097F]+', translated_text)
    if deva_matches:
        issues['devanagari'] = (False, f"Found {len(deva_matches)} Devanagari segment(s): {deva_matches[:3]}")
    else:
        issues['devanagari'] = (True, "Clean")

    # CHECK 2: Separator artifact leak
    sep_matches = re.findall(r'[\u2500-\u257F]{3,}', translated_text)
    sep_matches += re.findall(r'(?m)^[=\-\*]{5,}\s*$', translated_text)
    if sep_matches:
        issues['separators'] = (False, f"Found {len(sep_matches)} separator artifact(s) in output")
    else:
        issues['separators'] = (True, "Clean")

    # CHECK 3: Untranslated English block detection (5+ consecutive English words)
    # CHECK 3: Untranslated English block detection (5+ consecutive English words)
    english_runs = re.findall(r'\b[A-Za-z]{3,}(?:\s+[A-Za-z]{3,}){4,}\b', translated_text)
    real_english_runs = []
    for run in english_runs:
        words = run.split()
        lowercase_count = sum(1 for w in words if w[0].islower())
        if lowercase_count >= 3:
            real_english_runs.append(run[:80])
    if real_english_runs:
        issues['untranslated_english'] = (False, f"Found {len(real_english_runs)} likely untranslated run(s): {real_english_runs[:2]}")
    else:
        issues['untranslated_english'] = (True, "Clean")

    # CHECK 4: Overlap duplication check
    if prev_chunk_tail and len(prev_chunk_tail) > 20:
        prev_words = set(w.lower() for w in prev_chunk_tail.split()[-15:])
        curr_words = set(w.lower() for w in translated_text.split()[:15])
        overlap_count = len(prev_words & curr_words)
        similarity = overlap_count / max(len(prev_words), 1)
        if similarity > 0.6:
            issues['overlap_duplication'] = (False, f"High overlap similarity ({similarity:.0%}) — possible retranslation of context")
        else:
            issues['overlap_duplication'] = (True, f"OK ({similarity:.0%})")
    else:
        issues['overlap_duplication'] = (True, "No previous chunk")

    return issues


def summarise_issues(issues_dict):
    """Format QA results as compact string for console."""
    lines = []
    all_passed = True
    for check, (passed, detail) in issues_dict.items():
        icon = "✅" if passed else "⚠️ "
        if not passed:
            all_passed = False
        lines.append(f"   {icon} {check}: {detail}")
    return "\n".join(lines), all_passed


def clean_translation(text):
    # Remove thinking blocks
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    # Remove markdown code blocks
    text = re.sub(r'```\w*\n?', '', text)
    # Remove translation prefix labels
    text = re.sub(
        r'(Translation:|Hindi Translation:|Here\'s the translation:|Hindi:|Hinglish Translation:)',
        '', text, flags=re.IGNORECASE)
    # Remove triple quotes
    text = text.replace(chr(34)*3, '').replace(chr(39)*3, '')
    # Remove context bleed (old format with ━ separators)
    text = re.sub(r'CONTEXT \(last lines.*?\(Context ends.*?\n', '', text, flags=re.DOTALL)
    # Remove context bleed (new plain-text format)
    text = re.sub(r'\[PICHLI TRANSLATION KA AAKHRI HISSA.*?\[CONTEXT KHATAM.*?\]', '', text, flags=re.DOTALL)
    # Strip any prompt separator characters that leaked into output
    # ━ (U+2501), = repeated 5+, - repeated 5+, * repeated 5+
    text = re.sub(r'[\u2500-\u257F]{3,}', '', text)   # box-drawing chars
    text = re.sub(r'(?m)^[=\-\*]{5,}\s*$', '', text)  # ===== or ----- lines
    # Clean lines, preserve paragraph structure
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        stripped = line.strip()
        if stripped:
            cleaned_lines.append(stripped)
        elif cleaned_lines and cleaned_lines[-1] != '':
            cleaned_lines.append('')
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


# ──────────────────────────────────────────────────────────────────────────
# HUGGINGFACE ENGINE
# ──────────────────────────────────────────────────────────────────────────
class TranslationEngine:
    def __init__(self, model_name, target_lang, device="cuda", hf_token=None):
        self.model_name = model_name
        self.target_lang = target_lang
        self.device = device
        self.hf_token = hf_token
        self.model = None
        self.tokenizer = None
        self.translator = None
        self.model_type = self._detect_model_type(model_name)

        if not self.hf_token:
            try:
                from huggingface_hub import HfFolder
                self.hf_token = HfFolder.get_token()
            except:
                pass

        if self.hf_token:
            os.environ['HF_TOKEN'] = self.hf_token

        self.load_model()

    def _detect_model_type(self, model_name):
        m = model_name.lower()
        if 'nllb' in m: return 'nllb'
        elif 'indictrans' in m: return 'indictrans'
        elif 'opus-mt' in m or 'helsinki' in m: return 'opus'
        elif 'madlad' in m: return 'madlad'
        elif 'mbart' in m: return 'mbart'
        elif 'hy-mt' in m or 'hunyuan' in m: return 'hymt'
        elif 'translategemma' in m: return 'translategemma'
        else: return 'causal'

    def load_model(self):
        print(f"📥 Loading model: {self.model_name}")
        print(f"   Model type: {self.model_type}")
        try:
            if self.model_type in ['nllb', 'opus', 'mbart']:
                self._load_seq2seq_model()
            elif self.model_type == 'indictrans':
                self._load_indictrans_model()
            elif self.model_type == 'madlad':
                self._load_madlad_model()
            elif self.model_type == 'hymt':
                self._load_hymt_model()
            elif self.model_type == 'translategemma':
                self._load_translategemma_model()
            else:
                self._load_causal_model()
            print("✅ Model loaded successfully!")
        except Exception as e:
            print(f"❌ Failed to load model: {e}")
            raise

    def _load_seq2seq_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token,
            src_lang="eng_Latn" if self.model_type == 'nllb' else None)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.device != "cuda":
            self.model = self.model.to(self.device)
        self.translator = pipeline(
            "translation", model=self.model, tokenizer=self.tokenizer,
            src_lang="eng_Latn" if self.model_type == 'nllb' else "en",
            tgt_lang=self.target_lang if self.model_type == 'nllb' else None,
            max_length=1024, device=0 if self.device == "cuda" else -1)

    def _load_indictrans_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32)
        if self.device == "cuda":
            self.model = self.model.cuda()

    def _load_madlad_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)

    def _load_hymt_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token, trust_remote_code=True,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _load_translategemma_model(self):
        if not self.hf_token:
            raise ValueError("TranslateGemma is a gated model. Please login to HuggingFace first.")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _load_causal_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=self.hf_token)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name, token=self.hf_token,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def _filter_inputs(self, inputs):
        if 'token_type_ids' in inputs:
            del inputs['token_type_ids']
        return inputs

    def translate(self, text, tier="INTERMEDIATE", prev_translation="", overlap_words=0):
        if self.model_type in ['nllb', 'opus', 'mbart']:
            return self._translate_seq2seq(text)
        elif self.model_type == 'indictrans':
            return self._translate_indictrans(text)
        elif self.model_type == 'madlad':
            return self._translate_madlad(text)
        elif self.model_type == 'hymt':
            return self._translate_hymt(text)
        elif self.model_type == 'translategemma':
            return self._translate_translategemma(text)
        else:
            return self._translate_causal(text, tier, prev_translation, overlap_words)

    def _translate_seq2seq(self, text):
        result = self.translator(text, max_length=1024)
        return result[0]['translation_text']

    def _translate_indictrans(self, text):
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=512, num_beams=5, num_return_sequences=1)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def _translate_madlad(self, text):
        lang_code = self.target_lang.split('_')[0]
        tagged_text = f"<2{lang_code}> {text}"
        inputs = self.tokenizer(tagged_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=512, num_beams=4)
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

    def _translate_hymt(self, text):
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)
        prompt = f"Translate the following text to {lang_name}:\n{text}\n\nTranslation:"
        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024, temperature=0.3,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id,
                                          eos_token_id=self.tokenizer.eos_token_id)
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Translation:' in generated:
            return generated.split('Translation:')[-1].strip()
        return generated[len(prompt):].strip()

    def _translate_translategemma(self, text):
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)
        prompt = f"Translate the following text from English to {lang_name}:\n\n{text}\n\nTranslation:"
        inputs = self.tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=1024, temperature=0.3,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id)
        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if 'Translation:' in generated:
            return generated.split('Translation:')[-1].strip()
        return generated[len(prompt):].strip()

    def _translate_causal(self, text, tier, prev_translation="", overlap_words=0):
        prompts = TRANSLATION_PROMPTS[tier]
        lang_name = LANG_NAMES.get(self.target_lang, self.target_lang)

        scene_type = detect_scene_type(text)
        scene_context = SCENE_CONTEXT_LINES.get(scene_type, SCENE_CONTEXT_LINES['DAILY_LIFE'])
        overlap_tail = get_overlap_context(prev_translation, overlap_words)
        overlap_section = build_overlap_section(overlap_tail)

        user_msg = prompts['user'].format(
            target_lang=lang_name, chunk=text,
            scene_context=scene_context,
            overlap_section=overlap_section
        )
        full_prompt = f"{prompts['system']}\n\n{user_msg}"

        inputs = self.tokenizer(full_prompt, return_tensors="pt", padding=True, truncation=True, max_length=3072)
        inputs = self._filter_inputs(inputs)
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=2048, temperature=0.3,
                                          do_sample=True, pad_token_id=self.tokenizer.pad_token_id)

        generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "Translation:" in generated:
            return generated.split("Translation:")[-1].strip()
        return generated[len(full_prompt):].strip()


class TranslationGenerator:
    def __init__(self, model_name, target_lang, device="cuda", output_dir=".", tier="INTERMEDIATE",
                 chunk_size=550, hf_token=None, overlap_words=45):
        self.model_name = model_name
        self.target_lang = target_lang
        self.device = device
        self.output_dir = Path(output_dir)
        self.tier = tier
        self.chunk_size = chunk_size
        self.overlap_words = overlap_words
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = TranslationEngine(model_name, target_lang, device, hf_token)

    def translate_file(self, input_file):
        print(f"\n{'=' * 70}")
        print(f"🌐 TRANSLATION GENERATOR  v4")
        print(f"{'=' * 70}")
        print(f"📄 Input: {input_file}")
        print(f"🤖 Model: {self.model_name}")
        print(f"🌐 Target: {self.target_lang}")
        print(f"🎯 Quality: {self.tier}")
        print(f"🖥️ Device: {self.device}")
        print(f"🔀 Overlap: {self.overlap_words} words")
        print(f"{'=' * 70}\n")

        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()

        # Strip === === section markers
        lines = text.split('\n')
        cleaned = [l for l in lines if not (l.strip().startswith('===') and l.strip().endswith('==='))]
        text = '\n'.join(cleaned).strip()

        # ★ NEW: strip embedded page-number artifacts
        text = clean_source_text(text)

        orig_words = len(text.split())
        orig_chars = len(text)
        print(f"📊 Input: {orig_chars:,} chars, {orig_words:,} words")

        print(f"\n📦 Creating chunks ({self.chunk_size} words each)...")
        chunks = chunk_text(text, self.chunk_size)
        print(f"✅ Created {len(chunks)} chunks")

        print(f"\n🎯 STARTING TRANSLATION\n")

        translations = []
        devanagari_flags = []
        start_time = time.time()

        qa_report = []

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()
            scene_type = detect_scene_type(chunk)
            prev_translation = translations[-1] if translations else ""
            prev_tail = get_overlap_context(prev_translation, self.overlap_words) if prev_translation else ""

            print(f"\n{'=' * 50}")
            print(f"📄 Chunk {i}/{len(chunks)} | 🎭 Scene: {scene_type}")
            print(f"   Input: {len(chunk.split())} words, {len(chunk)} chars")

            try:
                translated = self.engine.translate(chunk, self.tier, prev_translation, self.overlap_words)
                translated = clean_translation(translated)

                issues = validate_translation(translated, prev_tail, chunk_index=i)
                qa_summary, all_passed = summarise_issues(issues)
                qa_report.append({"chunk": i, "scene": scene_type, "issues": issues})

                deva_flag = not issues['devanagari'][0]
                devanagari_flags.append(deva_flag)
                translations.append(translated)

                chunk_time = time.time() - chunk_start
                print(f"   Output: {len(translated)} chars | {chunk_time:.1f}s")
                if all_passed:
                    print(f"   ✅ All QA checks passed")
                else:
                    print(f"   🔍 QA Results:")
                    print(qa_summary)

                elapsed = time.time() - start_time
                avg = elapsed / i
                eta = (len(chunks) - i) * avg
                print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

            except Exception as e:
                print(f"   ❌ Error: {e}")
                translations.append(f"[TRANSLATION ERROR: {e}]")
                devanagari_flags.append(False)
                qa_report.append({"chunk": i, "scene": scene_type, "issues": {"error": (False, str(e))}})

        final_translation = "\n\n".join(translations)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        lang_code = self.target_lang.split('_')[0]
        output_file = self.output_dir / f"translation_{lang_code}_{timestamp}.txt"

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_translation)

        total_time = time.time() - start_time
        trans_chars = len(final_translation)
        deva_count = sum(devanagari_flags)

        print(f"\n{'=' * 70}")
        print(f"🎉 TRANSLATION COMPLETE!")
        print(f"{'=' * 70}")
        print(f"⏱️ Time: {total_time/60:.1f} minutes")
        print(f"📦 Chunks: {len(chunks)}")
        print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
        print(f"📝 Input: {orig_chars:,} chars")
        print(f"📝 Output: {trans_chars:,} chars")
        print(f"📊 Ratio: {trans_chars/orig_chars:.2f}x")
        if deva_count > 0:
            print(f"⚠️ Devanagari found in {deva_count}/{len(chunks)} chunks — review those chunks")
        else:
            print(f"✅ All chunks: Roman/Latin script only")
        print(f"💾 Output: {output_file}")
        print(f"{'=' * 70}")

        return str(output_file)


# ──────────────────────────────────────────────────────────────────────────
# OLLAMA ENGINE
# ──────────────────────────────────────────────────────────────────────────
class OllamaTranslationEngine:
    def __init__(self, model_name, target_lang, tier="INTERMEDIATE"):
        self.model_name = model_name
        self.target_lang = target_lang
        self.tier = tier
        self.lang_name = LANG_NAMES.get(target_lang, target_lang)

        print(f"📥 Initializing Ollama engine: {model_name}")
        print(f"   Target language: {self.lang_name}")

        try:
            import ollama
            self.client = ollama
            models = ollama.list()
            available = [m.get('name', '').split(':')[0] for m in models.get('models', [])]
            model_base = model_name.split(':')[0]

            if not any(model_base in m for m in available):
                print(f"⚠️ Model '{model_name}' not found locally. Pulling...")
                ollama.pull(model_name)
                print(f"✅ Model '{model_name}' pulled successfully!")
            else:
                print(f"✅ Model '{model_name}' is available!")
        except Exception as e:
            print(f"❌ Error initializing Ollama: {e}")
            raise

    def translate(self, text, tier=None, prev_translation="", overlap_words=45):
        if tier:
            self.tier = tier

        prompts = TRANSLATION_PROMPTS[self.tier]
        system_prompt = prompts['system']

        scene_type = detect_scene_type(text)
        scene_context = SCENE_CONTEXT_LINES.get(scene_type, SCENE_CONTEXT_LINES['DAILY_LIFE'])
        overlap_tail = get_overlap_context(prev_translation, overlap_words)
        overlap_section = build_overlap_section(overlap_tail)

        user_prompt = prompts['user'].format(
            target_lang=self.lang_name,
            chunk=text,
            scene_context=scene_context,
            overlap_section=overlap_section
        )

        max_retries = 3
        last_translation = None

        for attempt in range(max_retries):
            try:
                retry_suffix = ""
                if attempt > 0 and last_translation:
                    # Check why we're retrying
                    if has_devanagari(last_translation):
                        retry_suffix = "\n\nWARNING: Tumhara pichla response mein Devanagari characters the. SIRF Roman/Latin script use karo. Ek bhi Devanagari = FAIL."
                    else:
                        retry_suffix = "\n\nWARNING: Tumhara pichla response mein English sentences translate nahi hue the. HAR CHEEZ translate karo. Koi English sentence chhodna = FAIL."

                current_user_prompt = user_prompt + retry_suffix

                response = self.client.chat(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": current_user_prompt}
                    ],
                    options={"temperature": 0.3, "num_predict": 8192, "repeat_penalty": 1.15}
                )
                translation = response['message']['content']
                translation = clean_translation(translation)
                last_translation = translation

                # Check if retry is needed
                needs_retry = False

                # Check for Devanagari
                if has_devanagari(translation):
                    print(f"   ⚠️ Devanagari detected (attempt {attempt+1}/{max_retries}), retrying...")
                    needs_retry = True

                # Check for large untranslated English blocks (>30% English words)
                if not needs_retry:
                    words = translation.split()
                    # Expanded allowlist: proper nouns + naturally English words in Hinglish + common transliterations
                    _hinglish_allowlist = {
                        # Proper nouns / names (any story)
                        'holmes', 'watson', 'irene', 'adler', 'london', 'baker', 'street',
                        'bohemia', 'briony', 'lodge', 'norton', 'godfrey', 'moriarty',
                        'church', 'temple', 'park', 'avenue', 'langham', 'europe',
                        'doctor', 'king', 'queen', 'prince', 'princess',
                        # Titles
                        'mr', 'mrs', 'sir', 'miss', 'lord', 'lady', 'madam',
                        # Naturally English words used in Hinglish
                        'police', 'train', 'office', 'cab', 'plan', 'hotel', 'station',
                        'college', 'school', 'phone', 'mobile', 'camera', 'photo',
                        'message', 'letter', 'card', 'note', 'sign', 'signal',
                        'case', 'file', 'report', 'paper', 'news', 'story',
                        'road', 'lane', 'market', 'shop', 'door', 'room', 'hall',
                        'dress', 'coat', 'mask', 'ring', 'watch', 'glass',
                        'cigarette', 'pipe', 'smoke', 'fire', 'light',
                        'game', 'match', 'team', 'score', 'point',
                        # Scientific / technical
                        'iodoform', 'stethoscope', 'nitrate', 'silver',
                        # Common Hinglish transliterations (Roman script Hindi)
                        'nahi', 'kahi', 'bhai', 'yaar', 'karo', 'raha', 'wala',
                        'haan', 'abhi', 'acha', 'accha', 'theek', 'sahi',
                        'dekho', 'suno', 'chalo', 'jao', 'aao', 'bolo', 'batao',
                        'samajh', 'samjho', 'socho', 'padho', 'likho', 'rakho',
                        'kuch', 'kisi', 'koi', 'sab', 'sabse', 'bahut', 'zyada',
                        'woh', 'yeh', 'uska', 'uski', 'iska', 'iski', 'unka', 'unki',
                        'mera', 'meri', 'tera', 'teri', 'tumhara', 'tumhari',
                        'toh', 'bhi', 'hota', 'hoti', 'tha', 'thi', 'the',
                        'kar', 'kiya', 'kiye', 'ki', 'ke', 'ka', 'ko', 'se', 'mein', 'pe',
                        'liye', 'liya', 'diya', 'diye', 'gaya', 'gayi', 'gaye',
                        'aaya', 'aayi', 'aaye', 'hota', 'hoti', 'hote',
                        'pehle', 'baad', 'saath', 'andar', 'bahar', 'upar', 'neeche',
                        'seedha', 'ulta', 'saamne', 'peechhe', 'aage',
                        'baat', 'kaam', 'dost', 'ghar', 'raat', 'din', 'waqt', 'baar',
                        'cheez', 'jagah', 'taraf', 'haal', 'sawaal', 'jawaab',
                        'aadmi', 'aurat', 'ladka', 'ladki', 'insaan', 'banda', 'bandi',
                        'dimaag', 'dil', 'aankhein', 'haath', 'paer', 'saans',
                        'bilkul', 'ekdum', 'sachchi', 'pakka', 'seedhi',
                        'huzoor', 'sahab', 'bade', 'chhote', 'purane', 'naye',
                        'pata', 'matlab', 'lagta', 'lagti', 'chahiye', 'sakta', 'sakti',
                        'wali', 'waala', 'jaisa', 'jaisi', 'jaise',
                        'lekin', 'magar', 'kyunki', 'isliye', 'jab', 'tab',
                        'agar', 'toh', 'phir', 'fir', 'abhi', 'kabhi',
                        'poora', 'poori', 'thoda', 'thodi', 'zyada',
                        'kamaal', 'mazaa', 'maza', 'shukriya', 'maaf',
                        'bola', 'boli', 'bole', 'pucha', 'puchha', 'chilla',
                        'dekha', 'suna', 'mila', 'mili', 'mile',
                    }
                    english_word_count = sum(1 for w in words if re.match(r'^[a-zA-Z]{4,}$', w) and w.lower() not in _hinglish_allowlist)
                    english_ratio = english_word_count / max(len(words), 1)
                    if english_ratio > 0.50:
                        print(f"   ⚠️ High English ratio ({english_ratio:.0%}, attempt {attempt+1}/{max_retries}), retrying...")
                        needs_retry = True

                if not needs_retry or attempt == max_retries - 1:
                    return translation

            except Exception as e:
                if attempt == max_retries - 1:
                    print(f"❌ Ollama translation error: {e}")
                    raise
                print(f"   ⚠️ Attempt {attempt+1} failed: {e}, retrying...")

        return last_translation


class OllamaTranslationGenerator:
    def __init__(self, model_name, target_lang, output_dir=".", tier="INTERMEDIATE",
                 chunk_size=550, overlap_words=45):
        self.model_name = model_name
        self.target_lang = target_lang
        self.output_dir = Path(output_dir)
        self.tier = tier
        self.chunk_size = chunk_size
        self.overlap_words = overlap_words
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.engine = OllamaTranslationEngine(model_name, target_lang, tier)

    def translate_file(self, input_file):
        print(f"\n{'=' * 70}")
        print(f"🌐 OLLAMA TRANSLATION GENERATOR  v6.1")
        print(f"{'=' * 70}")
        print(f"📄 Input: {input_file}")
        print(f"🦙 Model: {self.model_name}")
        print(f"🌐 Target: {self.target_lang}")
        print(f"🎯 Quality: {self.tier}")
        print(f"🔀 Overlap: {self.overlap_words} words of previous translation as context")
        print(f"{'=' * 70}\n")

        with open(input_file, 'r', encoding='utf-8') as f:
            text = f.read()

        lines = text.split('\n')
        cleaned = [l for l in lines if not (l.strip().startswith('===') and l.strip().endswith('==='))]
        text = '\n'.join(cleaned).strip()

        # ★ NEW: strip embedded page-number artifacts
        text = clean_source_text(text)

        orig_words = len(text.split())
        orig_chars = len(text)
        print(f"📊 Input: {orig_chars:,} chars, {orig_words:,} words")

        print(f"\n📦 Creating chunks ({self.chunk_size} words each)...")
        chunks = chunk_text(text, self.chunk_size)
        print(f"✅ Created {len(chunks)} chunks\n")

        print(f"🎯 STARTING TRANSLATION\n")

        translations = []
        devanagari_flags = []
        start_time = time.time()

        qa_report = []  # collect per-chunk QA results

        for i, chunk in enumerate(chunks, 1):
            chunk_start = time.time()
            scene_type = detect_scene_type(chunk)
            prev_translation = translations[-1] if translations else ""
            prev_tail = get_overlap_context(prev_translation, self.overlap_words) if prev_translation else ""

            print(f"\n{'=' * 50}")
            print(f"📄 Chunk {i}/{len(chunks)} | 🎭 Scene: {scene_type}")
            print(f"   Input: {len(chunk.split())} words, {len(chunk)} chars")
            if self.overlap_words > 0 and prev_translation:
                tail_words = len(prev_tail.split())
                print(f"   🔀 Context: last ~{tail_words} words of prev translation injected")

            try:
                translated = self.engine.translate(chunk, self.tier, prev_translation, self.overlap_words)
                translated = clean_translation(translated)  # clean BEFORE validation

                # ── QA VALIDATION ─────────────────────────────────────────
                issues = validate_translation(translated, prev_tail, chunk_index=i)
                qa_summary, all_passed = summarise_issues(issues)
                qa_report.append({"chunk": i, "scene": scene_type, "issues": issues})

                deva_flag = not issues['devanagari'][0]
                devanagari_flags.append(deva_flag)
                translations.append(translated)

                chunk_time = time.time() - chunk_start
                print(f"   Output: {len(translated)} chars | {chunk_time:.1f}s")
                if all_passed:
                    print(f"   ✅ All QA checks passed")
                else:
                    print(f"   🔍 QA Results:")
                    print(qa_summary)

                elapsed = time.time() - start_time
                avg = elapsed / i
                eta = (len(chunks) - i) * avg
                print(f"   📈 Progress: {i/len(chunks)*100:.1f}% | ETA: {eta/60:.1f}m")

            except Exception as e:
                print(f"   ❌ Error: {e}")
                translations.append(f"[TRANSLATION ERROR: {e}]")
                devanagari_flags.append(False)
                qa_report.append({"chunk": i, "scene": scene_type, "issues": {"error": (False, str(e))}})

        final_translation = "\n\n".join(translations)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        lang_code = self.target_lang.split('_')[0]
        output_file = self.output_dir / f"translation_{lang_code}_{timestamp}.txt"

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(final_translation)

        total_time = time.time() - start_time
        trans_chars = len(final_translation)
        deva_count = sum(devanagari_flags)

        print(f"\n{'=' * 70}")
        print(f"🎉 TRANSLATION COMPLETE!")
        print(f"{'=' * 70}")
        print(f"⏱️  Time: {total_time/60:.1f} minutes")
        print(f"📦 Chunks: {len(chunks)}")
        print(f"⚡ Avg/chunk: {total_time/len(chunks):.1f}s")
        print(f"📝 Input: {orig_chars:,} chars")
        print(f"📝 Output: {trans_chars:,} chars")
        print(f"📊 Ratio: {trans_chars/orig_chars:.2f}x")
        if deva_count > 0:
            print(f"⚠️  Devanagari found in {deva_count}/{len(chunks)} chunks")
        else:
            print(f"✅ All chunks passed Roman-only script check")

        # Write QA report
        qa_report_file = self.output_dir / f"qa_report_{lang_code}_{timestamp}.txt"
        qa_failed_chunks = []
        with open(qa_report_file, 'w', encoding='utf-8') as qf:
            qf.write(f"QA REPORT — {timestamp}\n")
            qf.write(f"Model: {self.model_name} | Chunks: {len(chunks)}\n")
            qf.write("=" * 60 + "\n\n")
            for entry in qa_report:
                issues = entry['issues']
                failed = [k for k, (p, _) in issues.items() if not p]
                status = "✅ PASS" if not failed else f"⚠️  FAIL [{', '.join(failed)}]"
                qf.write(f"Chunk {entry['chunk']} ({entry['scene']}): {status}\n")
                if failed:
                    qa_failed_chunks.append(entry['chunk'])
                    for k in failed:
                        qf.write(f"  → {k}: {issues[k][1]}\n")
            qf.write(f"\nSUMMARY: {len(qa_failed_chunks)}/{len(chunks)} chunks had issues\n")
            if qa_failed_chunks:
                qf.write(f"Failed chunks: {qa_failed_chunks}\n")

        if qa_failed_chunks:
            print(f"⚠️  QA: {len(qa_failed_chunks)} chunk(s) had issues — see {qa_report_file}")
        else:
            print(f"✅ QA: All chunks passed all 5 checks")
        print(f"💾 Output: {output_file}")
        print(f"📋 QA Report: {qa_report_file}")
        print(f"{'=' * 70}")

        # Store qa_report for access after run
        self.last_qa_report = qa_report
        self.last_qa_file = str(qa_report_file)

        return str(output_file)


print("✅ Translation Engine v6.1 loaded!")
print("   → Generic scene detection (DEDUCTION / REVELATION / TENSION / BANTER / CONFRONTATION / ...)")
print("   → Source artifact cleaner (removes embedded page-number lines from any book)")
print("   → Gen-Z Hinglish ADVANCED v2 prompt — shorter, stronger, expanded cheatsheet")
print("   → Victorian-to-Hinglish cheatsheet (60+ words) + 3 before/after examples")
print("   → Overlap context | Temp 0.3 | repeat_penalty 1.15 | Auto-retry on English/Devanagari\n")
print("   → Full QA validation: Devanagari, Separators, English blocks, Overlap Dupe + auto-retry")

## 🌐 Step 5: Generate Translation
Run this cell to translate your uploaded file.

In [ ]:
# Create output directory
OUTPUT_DIR = "./translation_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Initialize the generator based on provider
print("🚀 Initializing Translation Generator...")
print(f"   Provider: {SELECTED_PROVIDER}")
print(f"   Model: {SELECTED_MODEL}")
print(f"   Overlap: {OVERLAP_WORDS} words")

if SELECTED_PROVIDER == "ollama":
    generator = OllamaTranslationGenerator(
        model_name=SELECTED_MODEL,
        target_lang=TARGET_LANGUAGE,
        output_dir=OUTPUT_DIR,
        tier=TRANSLATION_TIER,
        chunk_size=CHUNK_SIZE,
        overlap_words=OVERLAP_WORDS
    )
else:
    generator = TranslationGenerator(
        model_name=SELECTED_MODEL,
        target_lang=TARGET_LANGUAGE,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
        tier=TRANSLATION_TIER,
        chunk_size=CHUNK_SIZE,
        hf_token=HF_TOKEN,
        overlap_words=OVERLAP_WORDS
    )

# Translate
print(f"\n🌐 Starting translation...")
OUTPUT_FILE = generator.translate_file(UPLOADED_FILE)
print(f"\n✅ Translation file generated: {OUTPUT_FILE}")


## 📖 Step 6: Download Translation


In [ ]:
# Download the translated file
from google.colab import files

print("📥 Downloading your translated file...")
files.download(OUTPUT_FILE)
print("✅ Download started! Check your browser's download folder.")

## 💾 (Optional) Save to Google Drive
If you want to save the translation to your Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
import shutil

print("📂 Mounting Google Drive...")
drive.mount('/content/drive')

# Create output folder in Drive
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Translation_Output"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

# Copy file to Drive
drive_output_path = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(OUTPUT_FILE))
shutil.copy(OUTPUT_FILE, drive_output_path)

print(f"\n✅ Translation saved to Google Drive:")
print(f"   📁 {drive_output_path}")

---

## 📚 Quick Reference

### Supported Languages (NLLB Codes):
| Language | Code |
|----------|------|
| Hindi | `hin_Deva` |
| Bengali | `ben_Beng` |
| Tamil | `tam_Taml` |
| Telugu | `tel_Telu` |
| Marathi | `mar_Deva` |
| Gujarati | `guj_Gujr` |
| Spanish | `spa_Latn` |
| French | `fra_Latn` |
| German | `deu_Latn` |

### Recommended Models:
| Model | Best For | Speed |
|-------|----------|-------|
| `facebook/nllb-200-distilled-600M` | Fast multilingual | ⚡ Fast |
| `facebook/nllb-200-1.3B` | Better quality | 🔄 Medium |
| `ai4bharat/indictrans2-en-indic-1B` | Best EN→Hindi | 🔄 Medium |
| `google/madlad400-3b-mt` | Highest quality | 🐢 Slow |

### Quality Tiers:
- **BASIC**: Fast, good for simple texts
- **INTERMEDIATE**: Balanced quality and speed (recommended)
- **ADVANCED**: Best quality, preserves all nuances

### Tips:
- Use Colab GPU for faster translation
- For long texts, use smaller chunk sizes (200-300 words)
- NLLB models are best for multilingual translation